# Event Impact Modelling

Sample Code (Please work on other events)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [8]:
"""
Step 1 — Define Events & Windows
  Example: CPI Releases
You need:
  Event date & time
  Pre-event window
  Post-event window
"""
events = pd.DataFrame({
  "event": ["CPI"],
  "datetime": [pd.Timestamp("2023-06-13 08:30:00")]
})

PRE_EVENT = pd.Timedelta(days=5)
POST_EVENT = pd.Timedelta(days=5)

"""
Step 2 — Load & Prepare Price Data
  Example using multiple assets:
"""
tickers = ["SPY", "EURUSD=X", "ZN=F"]  # equity, FX, rates
prices = yf.download(
  tickers,
  start="2023-01-01",
  end="2023-12-31",
  interval="1d"
)["Close"]

returns = prices.pct_change().dropna()

"""
Step 3 — Extract Event Windows
  This aligns price action to event time, not calendar time.
"""
def extract_event_window(returns, event_time, pre, post):
  window = returns.loc[event_time - pre : event_time + post].copy()
  window["event_time"] = (window.index - event_time).days  # or minutes
  return window

# Apply to all events
event_windows = {}

for ticker in returns.columns:
  event_windows[ticker] = extract_event_window(returns[[ticker]], events.loc[0, "datetime"], PRE_EVENT, POST_EVENT)


"""
Step 4 — Compute Event Impact Metrics
  Cumulative returns, volatility, drawdowns etc.
"""
def cumulative_return(df):
  return (1 + df).cumprod() - 1

def rolling_volatility(df, window=3):
  return df.rolling(window).std()

def drawdown(series):
  cum = (1 + series).cumprod()
  peak = cum.cummax()
  return (cum - peak) / peak

"""
Step 5 — Event Study Output per Asset
"""
results = {}

for ticker, df in event_windows.items():
  r = df[ticker]
  results[ticker] = {
    "cum_return": cumulative_return(r),
    "volatility": rolling_volatility(r),
    "drawdown": drawdown(r)
  }

"""
Step 6 — Speed of Price Discovery (Key Desk Insight)
  This is where your project stands out.
  Time to Absorb Information
  Define:
    How fast does the asset reach X% of its total event move?
    - Assets with shorter absorption time price information faster.
"""
def time_to_absorb(cum_returns, threshold=0.8):
  total_move = cum_returns.iloc[-1]
  target = threshold * total_move
  return (cum_returns >= target).idxmax()

speed = {}

for ticker in results:
  speed[ticker] = time_to_absorb(results[ticker]["cum_return"])


"""
Step 7 — Cross-Asset Comparison
  Create a comparison table
"""
summary = pd.DataFrame({
  ticker: {
    "Max Return": results[ticker]["cum_return"].max(),
    "Max Volatility": results[ticker]["volatility"].max(),
    "Max Drawdown": results[ticker]["drawdown"].min(),
    "Speed": speed[ticker]
  }
  for ticker in results
}).T

print("Summary of Event Impacts:")
print(summary)

/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_48105/350677523.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(
[*********************100%***********************]  3 of 3 completed

Summary of Event Impacts:
         Max Return Max Volatility Max Drawdown                Speed
EURUSD=X   0.022275       0.005016    -0.002946  2023-06-16 00:00:00
SPY        0.031389       0.008129    -0.003406  2023-06-15 00:00:00
ZN=F      -0.001797       0.005942      -0.0061  2023-06-09 00:00:00



/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_48105/350677523.py:29: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna()
